# KU Prep Arena — vLLM Serving on Google Colab

Model: `Frank290350/ku-typhoon-v1-merged`  
Serves OpenAI-compatible API on port 8000, then exposes via Cloudflare tunnel and auto-updates Vercel.

### Before running:
Add these **Colab Secrets** (🔑 icon on the left sidebar):
- `VERCEL_TOKEN` — from https://vercel.com/account/tokens
- `VERCEL_PROJECT_ID` — from Vercel project settings
- `VERCEL_TEAM_ID` — leave blank if personal account
- `HF_TOKEN` — from https://huggingface.co/settings/tokens (needed if model is private)

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, os

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND')

# Install vLLM (takes ~3-5 min on first run)
print('\nInstalling vLLM...')
subprocess.run(['pip', 'install', '-q', 'vllm>=0.8.0'], check=True)
print('vLLM installed.')

# Download cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('\nDownloading cloudflared...')
    subprocess.run([
        'wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print('cloudflared ready.')
else:
    print('cloudflared already present.')

print('\n✅ All dependencies ready.')

In [ ]:
# ── Cell 2: Start vLLM → wait → tunnel → update Vercel ───────────────────────
import subprocess, time, re, json, urllib.request, os, threading

# ── Config ────────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    VERCEL_TOKEN      = userdata.get('VERCEL_TOKEN')      or ''
    VERCEL_PROJECT_ID = userdata.get('VERCEL_PROJECT_ID') or ''
    VERCEL_TEAM_ID    = userdata.get('VERCEL_TEAM_ID')    or ''
    HF_TOKEN          = userdata.get('HF_TOKEN')          or ''
except Exception:
    VERCEL_TOKEN      = os.environ.get('VERCEL_TOKEN', '')
    VERCEL_PROJECT_ID = os.environ.get('VERCEL_PROJECT_ID', '')
    VERCEL_TEAM_ID    = os.environ.get('VERCEL_TEAM_ID', '')
    HF_TOKEN          = os.environ.get('HF_TOKEN', '')

MODEL_ID   = 'Frank290350/ku-typhoon-v1-merged'
MODEL_NAME = 'ku_typhoon_v1_merged'   # served-model-name (must match Vercel AI_MODEL)
PORT       = 8000
LOG_FILE   = '/tmp/vllm.log'
CF_LOG     = '/tmp/cf.log'

# ── Helpers ────────────────────────────────────────────────────────────────────
def wait_for_health(url, retries=60, interval=5):
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=3) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        print(f'  waiting... ({i+1}/{retries})', end='\r')
        time.sleep(interval)
    return False

def tail_log(path, lines=5):
    try:
        with open(path) as f:
            data = f.read()
        return '\n'.join(data.strip().split('\n')[-lines:])
    except Exception:
        return '(empty)'

def _req(method, url, body=None):
    headers = {'Authorization': f'Bearer {VERCEL_TOKEN}', 'Content-Type': 'application/json'}
    data = json.dumps(body).encode() if body else None
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        return json.loads(e.read())

def vercel_url(path, **qs):
    if VERCEL_TEAM_ID:
        qs['teamId'] = VERCEL_TEAM_ID
    q = '&'.join(f'{k}={v}' for k, v in qs.items())
    return f'https://api.vercel.com{path}{"?" + q if q else ""}'

def update_vercel_env(key, value):
    if not VERCEL_TOKEN or not VERCEL_PROJECT_ID:
        print('[SKIP] No VERCEL_TOKEN / VERCEL_PROJECT_ID — skipping Vercel update')
        return False

    targets = ['production', 'preview', 'development']
    existing = _req('GET', vercel_url(f'/v9/projects/{VERCEL_PROJECT_ID}/env'))
    existing_ids = [e['id'] for e in existing.get('envs', []) if e.get('key') == key]

    if existing_ids:
        for env_id in existing_ids:
            resp = _req('PATCH', vercel_url(f'/v9/projects/{VERCEL_PROJECT_ID}/env/{env_id}'),
                {'value': value, 'target': targets})
            if 'error' in resp:
                print(f'[Vercel] PATCH failed: {resp["error"]}')
                return False
        print(f'[Vercel] ✅ Updated {key}')
    else:
        resp = _req('POST', vercel_url(f'/v10/projects/{VERCEL_PROJECT_ID}/env'),
            {'key': key, 'value': value, 'type': 'plain', 'target': targets})
        if 'error' in resp:
            print(f'[Vercel] POST failed: {resp["error"]}')
            return False
        print(f'[Vercel] ✅ Created {key}')

    return True

def trigger_redeploy():
    if not VERCEL_TOKEN or not VERCEL_PROJECT_ID:
        return
    resp = _req('GET', vercel_url('/v6/deployments',
        projectId=VERCEL_PROJECT_ID, limit=5, target='production'))
    deps = resp.get('deployments', [])
    print(f'[Vercel] Found {len(deps)} deployment(s)')

    dep_id = None
    for dep in deps:
        dep_id = dep.get('uid') or dep.get('id')
        state  = dep.get('state', dep.get('readyState', ''))
        print(f'  → {dep_id} state={state}')
        if dep_id:
            break

    if not dep_id:
        print('[Vercel] ⚠️  No deployment found — redeploy manually from dashboard')
        return

    r = _req('POST', vercel_url('/v13/deployments', forceNew=1),
        {'deploymentId': dep_id, 'name': 'ku-prep-arena', 'target': 'production'})
    if 'error' in r:
        print(f'[Vercel] Redeploy error: {r["error"]}')
    else:
        new_id = r.get('id') or r.get('uid', '?')
        print(f'[Vercel] ✅ Redeploy triggered → {new_id}')

# ── Step 1: Start vLLM ────────────────────────────────────────────────────────
env = os.environ.copy()
if HF_TOKEN:
    env['HF_TOKEN'] = HF_TOKEN
    env['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print(f'Starting vLLM with {MODEL_ID} ...')
vllm_proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--served-model-name', MODEL_NAME,
    '--port', str(PORT),
    '--host', '0.0.0.0',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.90',
    '--dtype', 'float16',           # T4 does NOT support bfloat16
    '--enforce-eager',              # disable CUDA graph → frees ~2GB VRAM
    '--trust-remote-code',
    '--max-num-seqs', '4',
], stdout=open(LOG_FILE, 'w'), stderr=subprocess.STDOUT, env=env)

print(f'vLLM PID: {vllm_proc.pid}  |  log: {LOG_FILE}')
print('Waiting for vLLM to be ready (may take 3-8 min)...')

_stop = threading.Event()
def _stream_log():
    last = ''
    while not _stop.is_set():
        t = tail_log(LOG_FILE, 1)
        if t != last:
            print(' ', t, ' ' * 20, end='\r')
            last = t
        time.sleep(2)
threading.Thread(target=_stream_log, daemon=True).start()

if not wait_for_health(f'http://localhost:{PORT}/health'):
    _stop.set()
    print('\n[ERROR] vLLM did not start. Full log:')
    print(tail_log(LOG_FILE, 60))
    raise RuntimeError('vLLM failed to start')

_stop.set()
print(f'\n✅ vLLM ready on port {PORT}')

# ── Step 2: Start Cloudflare tunnel ──────────────────────────────────────────
print('\nStarting Cloudflare tunnel...')
cf_proc = subprocess.Popen([
    '/usr/local/bin/cloudflared', 'tunnel',
    '--url', f'http://localhost:{PORT}',
    '--protocol', 'http2',
    '--logfile', CF_LOG,
    '--loglevel', 'info',
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

TUNNEL_URL = ''
for _ in range(30):
    try:
        log_content = open(CF_LOG).read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_content)
        if m:
            TUNNEL_URL = m.group(0)
            break
    except Exception:
        pass
    time.sleep(2)

if not TUNNEL_URL:
    print('[ERROR] Could not find tunnel URL. CF log:')
    print(tail_log(CF_LOG, 10))
    raise RuntimeError('Tunnel URL not found')

print(f'\n✅ Tunnel URL: {TUNNEL_URL}')
print(f'   AI_BASE_URL: {TUNNEL_URL}/v1')
print(f'   AI_MODEL:    {MODEL_NAME}')

# ── Step 3: Update Vercel ─────────────────────────────────────────────────────
print('\nUpdating Vercel environment...')
ok = update_vercel_env('AI_BASE_URL', f'{TUNNEL_URL}/v1')
if ok:
    trigger_redeploy()
else:
    print(f'\n⚠️  Auto-update failed. Set manually in Vercel dashboard:')
    print(f'   AI_BASE_URL = {TUNNEL_URL}/v1')

print("""
═══════════════════════════════════════════════════
 KU Prep Arena — vLLM server is LIVE
 Keep this cell running (do NOT close the tab)
═══════════════════════════════════════════════════
""")

try:
    i = 0
    while True:
        time.sleep(300)
        i += 1
        print(f'[{i*5} min] still running — vLLM PID {vllm_proc.pid}')
except KeyboardInterrupt:
    print('\nStopping...')
    vllm_proc.terminate()
    cf_proc.terminate()